# Create fishtank scripts

Generates `SAMPLE_DIR/fishtank/` for this `lineage_tracing/lineage` experiment — the fishtank folder skeleton, shared reference files, and every run script (cellpose, detect-spots, decode-spots, mosaics) — replacing `07_create_merlin_scripts.ipynb` (used by the MERlin-based variants). Run after notebook 05 (`color_usage`/`decoding_strategy`) and notebook 02 (positions).

The lineage and merfish acquisitions share one sample: once split into acquisition-type subfolders, they're siblings under a common sample directory:
```
<sample_id>/
  merfish/   <- MERFISH_SAMPLE_DIR
  lineage/   <- SAMPLE_DIR (this notebook)
    fishtank/
```
`SAMPLE_NAME`/`IMAGING_DIR` are auto-detected from the folder structure (`resolve_sample_identity`), same as notebook 06 -- `IMAGING_DIR` is `""` in today's flat, unsplit layout (both acquisitions share this same folder sequentially, so `MERFISH_SAMPLE_DIR` resolves to `SAMPLE_DIR` itself instead), or `"lineage"` automatically once split into sibling folders. `SAMPLE_NAME` is the TRUE top-level experiment id, shared by both sibling acquisitions -- since notebook 02 in each acquisition now names its own positions file `positions_{SAMPLE_NAME}_{IMAGING_DIR}.txt` (`positions_file_tag`), each side's file already has its own acquisition-type tag (`_merfish` / `_lineage`) baked in, so the two never collide and no rename-on-copy is needed below.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent.parent
SAMPLE_DIR   = MERCI_DIR.parent
METADATA_DIR = SAMPLE_DIR / "metadata"
FISHTANK_DIR = SAMPLE_DIR / "fishtank"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.dave import count_positions
from MERci.acquisition.fishtank_config import (
    create_fishtank_folder_skeleton, copy_fishtank_reference_files,
    FishtankScriptsSpec, create_fishtank_scripts,
)
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.display        import display_file

# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 02-05 used (see notebook 02's docstring for why this isn't SAMPLE_DIR.name).
# It is shared by both sibling acquisitions (this one and the merfish one).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
LINEAGE_LIB_VERSION = "v2"                            # must match notebook 05

# IMAGING_DIR is auto-detected from the folder structure (resolve_sample_identity)
# instead of typed by hand -- "" = flat, unsplit layout (merfish and lineage
# share this one folder sequentially, so the merfish acquisition's own
# SAMPLE_DIR is this same folder); once split into sibling merfish/lineage
# folders, this becomes "lineage" automatically. Override manually below only
# if that detection picked the wrong thing.
# IMAGING_DIR = "lineage"
MERFISH_SAMPLE_DIR = (SAMPLE_DIR.parent / "merfish") if IMAGING_DIR else SAMPLE_DIR

# POSITIONS_TAG/MERFISH_POSITIONS_TAG are what each side's own notebook 02
# actually named its positions file with (positions_file_tag): SAMPLE_NAME
# alone in the flat layout (both acquisitions share one file, no tag needed),
# or SAMPLE_NAME_IMAGING_DIR once split -- "merfish" for the sibling
# acquisition's own subfolder, matching MERFISH_SAMPLE_DIR above.
POSITIONS_TAG         = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
MERFISH_IMAGING_DIR   = "merfish" if IMAGING_DIR else ""
MERFISH_POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, MERFISH_IMAGING_DIR)

print(f"SAMPLE_DIR         : {SAMPLE_DIR}")
print(f"SAMPLE_NAME        : {SAMPLE_NAME}")
print(f"IMAGING_DIR        : {IMAGING_DIR!r}")
print(f"POSITIONS_TAG      : {POSITIONS_TAG}")
print(f"MERFISH_SAMPLE_DIR : {MERFISH_SAMPLE_DIR}")
print(f"MERFISH_POSITIONS_TAG : {MERFISH_POSITIONS_TAG}")
if not MERFISH_SAMPLE_DIR.is_dir():
    print(f"WARNING: {MERFISH_SAMPLE_DIR} does not exist -- set it manually if your "
          f"sibling acquisition folder is named differently.")

## Folder skeleton + shared reference files

In [2]:
create_fishtank_folder_skeleton(FISHTANK_DIR, MERCI_DIR)
copy_fishtank_reference_files(LINEAGE_LIB_VERSION, MERCI_DIR, FISHTANK_DIR)
print(f"fishtank/ ready at {FISHTANK_DIR}")

fishtank/ ready at c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank


## Copy the merfish positions file alongside the lineage one

The generated `generate_mosaics.slurm` reads both acquisitions' positions files from ONE folder (`../../positions/` relative to `fishtank/scripts/`, i.e. `SAMPLE_DIR/positions/`) — matching the reference example's own convention. Copies the merfish positions file in; the lineage one already exists (notebook 02). Each side's own notebook 02 already names its file with its own acquisition-type tag (`positions_file_tag`), so the copy is a straight copy under its own name — no rename needed (in the flat/unsplit layout both tags are empty and resolve to the same file, so the copy is a no-op).

In [ ]:
LINEAGE_POSITIONS_NAME     = f"positions_{POSITIONS_TAG}.txt"
MERFISH_POSITIONS_SRC_NAME = f"positions_{MERFISH_POSITIONS_TAG}.txt"
# Each side's own tag already disambiguates the two filenames (empty tag ->
# same file in the flat layout, distinct tags -> distinct files once split),
# so the copy destination just reuses the source's own name.
MERFISH_POSITIONS_NAME     = MERFISH_POSITIONS_SRC_NAME

lineage_positions_path = SAMPLE_DIR / "positions" / LINEAGE_POSITIONS_NAME
merfish_positions_src  = MERFISH_SAMPLE_DIR / "positions" / MERFISH_POSITIONS_SRC_NAME
merfish_positions_dst  = SAMPLE_DIR / "positions" / MERFISH_POSITIONS_NAME

if not lineage_positions_path.exists():
    print(f"WARNING: {lineage_positions_path} not found -- run notebook 02 first.")
if merfish_positions_src.exists():
    if merfish_positions_src.resolve() == merfish_positions_dst.resolve():
        print(f"merfish positions already in place (flat layout): {merfish_positions_dst}")
    else:
        shutil.copy2(merfish_positions_src, merfish_positions_dst)
        print(f"Copied merfish positions: {merfish_positions_dst}")
else:
    print(f"WARNING: {merfish_positions_src} not found -- copy the merfish acquisition's "
          f"positions file to {merfish_positions_dst} manually before running generate_mosaics.slurm.")

## Script parameters

`FishtankScriptsSpec` — every field overridable, defaulting to the reference experiment's verified values (see `fishtank_config.py`'s docstrings). FOV counts are auto-derived from each acquisition's own positions file. `REF_SERIES` must match one of the `"beads"`-tagged rows you entered in notebook 05's `COLOR_USAGE_ROWS`.

In [ ]:
N_FOVS_LINEAGE = count_positions(lineage_positions_path) if lineage_positions_path.exists() else 0
N_FOVS_MERFISH = count_positions(merfish_positions_src) if merfish_positions_src.exists() else 0
print(f"N_FOVS_LINEAGE = {N_FOVS_LINEAGE}")
print(f"N_FOVS_MERFISH = {N_FOVS_MERFISH}")

REF_SERIES = ""   # e.g. "data/....tif" -- the beads/registration-reference series
                    # from notebook 05's COLOR_USAGE_ROWS

spec = FishtankScriptsSpec(
    n_fovs_lineage = N_FOVS_LINEAGE,
    n_fovs_merfish = N_FOVS_MERFISH,
)

## Generate the scripts

In [5]:
written = create_fishtank_scripts(
    spec, FISHTANK_DIR,
    experiment_label      = SAMPLE_NAME,
    lineage_input         = str(SAMPLE_DIR / "data"),
    merfish_input          = str(MERFISH_SAMPLE_DIR / "data"),
    mosaic_lineage_input   = str(SAMPLE_DIR / "data"),
    mosaic_merfish_input   = str(MERFISH_SAMPLE_DIR / "data"),
    lineage_positions      = f"../../positions/{LINEAGE_POSITIONS_NAME}",
    merfish_positions      = f"../../positions/{MERFISH_POSITIONS_NAME}",
    color_usage_lineage    = f"../../metadata/color_usage_{SAMPLE_NAME}.csv",
    color_usage_merfish    = f"../../metadata/color_usage_{SAMPLE_NAME}_mf.csv",
    decoding_strategy      = f"../../metadata/decoding_strategy_{SAMPLE_NAME}.csv",
    ref_series             = REF_SERIES,
)

for name, path in written.items():
    print(f"Saved: {path}")
    display_file(path)

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\cellpose_ft.slurm
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\cellpose_ft_mf.slurm
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\detect_spots_ft.slurm
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\decode_spots_ft.slurm
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\generate_mosaics.slurm
